In [0]:
# Caminho do arquivo de teste na Landing Zone
arquivo_localidade = (
    "/Volumes/workspace/bronze/landing/"
    "Localidade_DadosAbertos_20250512.csv"
)

print(arquivo_localidade)

In [0]:
# Inspeção das primeiras linhas do arquivo sem aplicar schema
df_preview = (
    spark.read
    .text(arquivo_localidade)
    .limit(3)
)

display(df_preview)

In [0]:
# Leitura estruturada do arquivo de Localidade
df_localidade = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .option("inferSchema", "false")
    .csv(arquivo_localidade)
)

display(df_localidade.limit(5))

In [0]:
# Validação da estrutura do arquivo de Localidade
print(f"Quantidade de colunas: {len(df_localidade.columns)}")

print("\nColunas identificadas:")
for coluna in df_localidade.columns:
    print(f"- {coluna}")

print("\nSchema:")
df_localidade.printSchema()

In [0]:
# Caminhos dos demais arquivos da Landing Zone
arquivos_landing = {
    "Acidentes": "/Volumes/workspace/bronze/landing/Acidentes_DadosAbertos_20250512.csv",
    "TipoVeiculo": "/Volumes/workspace/bronze/landing/TipoVeiculo_DadosAbertos_20250512.csv",
    "Vitimas": "/Volumes/workspace/bronze/landing/Vitimas_DadosAbertos_20250512.csv"
}

# Leitura apenas das 2 primeiras linhas brutas de cada arquivo:
# cabeçalho + primeiro registro
for nome, caminho in arquivos_landing.items():
    print(f"\n===== {nome} =====")
    
    linhas = (
        spark.read
        .text(caminho)
        .limit(2)
        .collect()
    )
    
    for linha in linhas:
        print(linha["value"])

In [0]:
# Arquivos da Landing Zone
arquivos_origem = {
    "localidade": arquivo_localidade,
    "acidentes": arquivos_landing["Acidentes"],
    "tipo_veiculo": arquivos_landing["TipoVeiculo"],
    "vitimas": arquivos_landing["Vitimas"]
}

# Leitura dos cabeçalhos, mantendo todos os campos como string
for nome, caminho in arquivos_origem.items():
    df = (
        spark.read
        .option("header", "true")
        .option("sep", ";")
        .option("encoding", "UTF-8")
        .option("inferSchema", "false")
        .csv(caminho)
    )

    print(f"\n===== {nome.upper()} =====")
    print(f"Quantidade de colunas: {len(df.columns)}")
    print("Colunas:")
    print(df.columns)

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col

# Função de leitura padronizada dos arquivos CSV da Landing Zone
def ler_csv_bronze(caminho):
    return (
        spark.read
        .option("header", "true")
        .option("sep", ";")
        .option("encoding", "UTF-8")
        .option("inferSchema", "false")
        .csv(caminho)
        .withColumn("_data_ingestao", current_timestamp())
        .withColumn("_arquivo_origem", col("_metadata.file_path"))
        .withColumn("_sistema_origem", lit("RENAEST/SENATRAN"))
    )

# DataFrames Bronze
df_bronze_localidade = ler_csv_bronze(arquivos_origem["localidade"])
df_bronze_acidentes = ler_csv_bronze(arquivos_origem["acidentes"])
df_bronze_tipo_veiculo = ler_csv_bronze(arquivos_origem["tipo_veiculo"])
df_bronze_vitimas = ler_csv_bronze(arquivos_origem["vitimas"])

print("DataFrames Bronze preparados com sucesso.")

In [0]:
# Validação do metadado de origem antes da gravação das tabelas
df_bronze_tipo_veiculo.select(
    "_arquivo_origem",
    "_sistema_origem",
    "_data_ingestao"
).limit(1).display()

In [0]:

# Persistência da primeira tabela Bronze em formato Delta
(
    df_bronze_localidade.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.localidade")
)

print("Tabela workspace.bronze.localidade gravada com sucesso.")

In [0]:
# Persistência da tabela Tipo Veículo na camada Bronze
(
    df_bronze_tipo_veiculo.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.tipo_veiculo")
)

print("Tabela workspace.bronze.tipo_veiculo gravada com sucesso.")

In [0]:
# Persistência da tabela Acidentes na camada Bronze
(
    df_bronze_acidentes.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.acidentes")
)

print("Tabela workspace.bronze.acidentes gravada com sucesso.")

In [0]:
# Persistência da tabela Vítimas na camada Bronze
(
    df_bronze_vitimas.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.vitimas")
)

print("Tabela workspace.bronze.vitimas gravada com sucesso.")

In [0]:
# Validação das tabelas persistidas na camada Bronze
tabelas_bronze = [
    "workspace.bronze.localidade",
    "workspace.bronze.tipo_veiculo",
    "workspace.bronze.acidentes",
    "workspace.bronze.vitimas"
]

for tabela in tabelas_bronze:
    df = spark.table(tabela)
    print(f"{tabela}: {len(df.columns)} colunas")